# 02 · Data Cleaning and Feature Engineering

**Purpose:** turn the raw `Tweets.csv` into a clean, validated analytical dataset.

`data/raw/Tweets.csv` ➜ **cleaning + label correction + feature engineering** ➜ `data/processed/cleaned_tweets.csv`

Every rule below is driven by a finding from notebook 01 and checked with assertions. The raw file is never overwritten; all steps work on a copy called `clean_df`.

## 0. Setup

In [1]:
import sys
from pathlib import Path

# Make the project's src/ package importable from any folder inside the repository
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "__init__.py").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import CLEAN_DATA_PATH
from src.data import correct_airline_labels, load_raw_tweets
from src.preprocessing import mask_user_handles

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [2]:
# Always start from the untouched raw file
df = load_raw_tweets()

# All cleaning happens on a copy so the raw DataFrame stays available for comparison
clean_df = df.copy()

print(f"Raw dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Raw dataset loaded: 14,640 rows x 15 columns


## 1. Initial Verification

A quick look at the starting point: shape, columns, missing values and duplicates.

In [3]:
print(f"Shape: {clean_df.shape}")
print(f"Exact duplicate rows: {clean_df.duplicated().sum():,}")
print(f"Duplicate tweet IDs: {clean_df.duplicated(subset='tweet_id').sum():,}")
print("\nColumns:")
for column_number, column_name in enumerate(clean_df.columns, start=1):
    print(f"{column_number:>2}. {column_name}")

Shape: (14640, 15)
Exact duplicate rows: 36
Duplicate tweet IDs: 155

Columns:
 1. tweet_id
 2. airline_sentiment
 3. airline_sentiment_confidence
 4. negativereason
 5. negativereason_confidence
 6. airline
 7. airline_sentiment_gold
 8. name
 9. negativereason_gold
10. retweet_count
11. text
12. tweet_coord
13. tweet_created
14. tweet_location
15. user_timezone


In [4]:
clean_df.isna().sum().to_frame("missing_values")

,missing_values
tweet_id,0
airline_sentiment,0
airline_sentiment_confidence,0
negativereason,5462
negativereason_confidence,4118
airline,0
airline_sentiment_gold,14600
name,0
negativereason_gold,14608
retweet_count,0


## 2. Remove Unnecessary Columns

Each column is evaluated before removal:

| Column | Missing | Decision | Reason |
|---|---|---|---|
| `airline_sentiment_gold` | ~99.7% | Remove | Only 40 rows have a value. It is an auditing field from the original labelling process and too sparse to use. The main `airline_sentiment` label is complete. |
| `negativereason_gold` | ~99.8% | Remove | Only 32 rows have a value. The complete `negativereason` column already holds the complaint category. |
| `tweet_coord` | ~93.0% | Remove | GPS coordinates exist for only ~7% of tweets, which is too sparse for reliable geographic analysis. None of the business questions need it. |
| `name` | 0% | Remove | This is the Twitter username. The analysis is at airline level, so the column adds no value and is avoidable personal data. |

All other columns are kept. `text` is retained so that the most-retweeted tweets can be shown in context, and because notebooks 06–10 use it as the input to the text models.

In [5]:
columns_to_remove = ["airline_sentiment_gold", "negativereason_gold", "tweet_coord", "name"]

# Show the evidence (missing %) behind each removal decision
removal_evidence = (clean_df[columns_to_remove].isna().mean() * 100).round(2).to_frame("missing_percentage")
display(removal_evidence)

clean_df = clean_df.drop(columns=columns_to_remove)
print(f"Shape after removing {len(columns_to_remove)} columns: {clean_df.shape}")

,missing_percentage
airline_sentiment_gold,99.73
negativereason_gold,99.78
tweet_coord,93.04
name,0.00


Shape after removing 4 columns: (14640, 11)


## 3. Sentiment Conflict Analysis

Before resolving duplicate tweet IDs, check whether any ID carries **more than one sentiment label**. This shows why a clear, documented rule is needed to choose one record per tweet.

In [6]:
# Number of distinct sentiment labels attached to each tweet_id
sentiment_labels_per_id = clean_df.groupby("tweet_id")["airline_sentiment"].nunique()
conflicting_tweet_ids = sentiment_labels_per_id[sentiment_labels_per_id > 1].index

conflict_columns = [
    "tweet_id", "airline", "airline_sentiment", "airline_sentiment_confidence",
    "negativereason", "negativereason_confidence", "text",
]

conflicting_records = (
    clean_df.loc[clean_df["tweet_id"].isin(conflicting_tweet_ids), conflict_columns]
    .sort_values(["tweet_id", "airline_sentiment_confidence"], ascending=[True, False])
)

print(f"Total conflicting tweet IDs: {len(conflicting_tweet_ids):,}")
print(f"Rows involved in conflicts: {len(conflicting_records):,}")
conflicting_records.assign(text=conflicting_records["text"].map(mask_user_handles))

Total conflicting tweet IDs: 18
Rows involved in conflicts: 36


,tweet_id,airline,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,text
12180,570268326250745856,American,negative,0.6670,Cancelled Flight,0.6670,@AmericanAir my flight to DFW from LIT on my way to PDX tomorrow was Cancelled Flighted. Can you help me?
12019,570268326250745856,American,neutral,0.6304,NaN,0.0000,@AmericanAir my flight to DFW from LIT on my way to PDX tomorrow was Cancelled Flighted. Can you help me?
12016,570268875872473088,American,positive,0.6907,NaN,0.0000,@AmericanAir I was happy to purchase the upgrade. If only it was avail on my next flight.
12177,570268875872473088,American,negative,0.6432,Flight Booking Problems,0.3331,@AmericanAir I was happy to purchase the upgrade. If only it was avail on my next flight.
12172,570270435478122497,American,positive,1.0000,NaN,NaN,@AmericanAir Thanks so much!
12011,570270435478122497,American,neutral,0.6266,NaN,NaN,@AmericanAir Thanks so much!
12165,570271574462369793,American,positive,1.0000,NaN,NaN,"@AmericanAir we are off to Kax premium. Hoping this flight is better food, TV now I know how to work it and service..."
12004,570271574462369793,American,neutral,0.6270,NaN,NaN,"@AmericanAir we are off to Kax premium. Hoping this flight is better food, TV now I know how to work it and service..."
12164,570271896887017472,American,neutral,0.6710,NaN,NaN,@AmericanAir thanks for the info Is there a number I can call to speak to a person? It's going to take an hour to ty...
12003,570271896887017472,American,positive,0.3519,NaN,0.0000,@AmericanAir thanks for the info Is there a number I can call to speak to a person? It's going to take an hour to ty...


**Observation:** for each conflicting ID, the repeated rows hold the same tweet text but different labels and confidence scores. They look like repeated annotations of the same tweet. The highest-confidence annotation is the most defensible one to keep (step 5).

## 4. Remove Completely Duplicate Rows

Exact duplicates are checked **after** the four unused columns are removed. Rows that differed only in those dropped columns (for example the `_gold` audit fields) are now identical for analysis. That is why slightly more rows are removed here than the raw-file exact-duplicate count in notebook 01.

In [7]:
rows_before = len(clean_df)
clean_df = clean_df.drop_duplicates()
rows_after = len(clean_df)

print(f"Rows before: {rows_before:,}")
print(f"Rows after:  {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,}")

Rows before: 14,640
Rows after:  14,601
Rows removed: 39


## 5. Resolve Duplicate Tweet IDs

A `tweet_id` should uniquely represent one tweet. For IDs that still appear more than once, the record kept is chosen by:

1. highest `airline_sentiment_confidence`
2. then highest `negativereason_confidence`

A stable sort keeps the result deterministic when confidences tie. The original row order is then restored and the index is reset.

In [8]:
rows_before_id_resolution = len(clean_df)
print(f"Duplicate tweet IDs before resolution: {clean_df.duplicated(subset='tweet_id').sum():,}")

clean_df = (
    clean_df
    # Highest confidence first; missing negativereason_confidence is ranked last
    .sort_values(
        by=["airline_sentiment_confidence", "negativereason_confidence"],
        ascending=[False, False],
        na_position="last",
        kind="mergesort",
    )
    # Keep the first (highest-confidence) record for each tweet_id
    .drop_duplicates(subset="tweet_id", keep="first")
    # Restore the original file order, then create a clean 0..n-1 index
    .sort_index()
    .reset_index(drop=True)
)

print(f"Rows before: {rows_before_id_resolution:,}")
print(f"Rows after:  {len(clean_df):,}")
print(f"Rows removed: {rows_before_id_resolution - len(clean_df):,}")

Duplicate tweet IDs before resolution: 116
Rows before: 14,601
Rows after:  14,485
Rows removed: 116


In [9]:
# Verification
remaining_exact_duplicates = clean_df.duplicated().sum()
remaining_duplicate_ids = clean_df.duplicated(subset="tweet_id").sum()

print(f"Remaining exact duplicates: {remaining_exact_duplicates}")
print(f"Remaining duplicate tweet IDs: {remaining_duplicate_ids}")

assert remaining_exact_duplicates == 0, "Exact duplicates remain"
assert remaining_duplicate_ids == 0, "Duplicate tweet IDs remain"

Remaining exact duplicates: 0
Remaining duplicate tweet IDs: 0


In [10]:
# Show which label was kept for each previously conflicting tweet ID
resolved_conflicts = clean_df.loc[
    clean_df["tweet_id"].isin(conflicting_tweet_ids),
    ["tweet_id", "airline", "airline_sentiment", "airline_sentiment_confidence", "negativereason"],
]
print(f"Conflicting IDs now represented by exactly one record: {len(resolved_conflicts):,}")
resolved_conflicts.head(10)

Conflicting IDs now represented by exactly one record: 18


,tweet_id,airline,airline_sentiment,airline_sentiment_confidence,negativereason
11881,570307312675651585,American,positive,0.6604,NaN
11896,570298285350629376,American,positive,0.7012,NaN
11916,570284482588246016,American,neutral,0.6593,NaN
11920,570282469791911936,American,neutral,1.0000,NaN
11923,570280641771790336,American,neutral,1.0000,NaN
11934,570275726483542016,American,neutral,1.0000,NaN
11951,570268875872473088,American,positive,0.6907,NaN
11984,570300355843661824,American,neutral,1.0000,NaN
11985,570300262302289920,American,positive,1.0000,NaN
11991,570296616688750592,American,neutral,1.0000,NaN


## 6. Missing Value Treatment

Rows are **not** dropped wholesale. `dropna()` on the full table would delete every positive and neutral tweet, because those tweets have no complaint reason. Each column is instead treated according to what its missing values mean:

| Column | Fill value | Why |
|---|---|---|
| `negativereason` | `Not Applicable` | Notebook 01 showed it is missing **only** for non-negative tweets. There is no complaint to record, so the value is "not applicable" rather than unknown. |
| `negativereason_confidence` | `0` | No complaint reason means zero confidence in one. The source already uses `0` for many non-negative tweets, so this keeps the column consistent. |
| `tweet_location` | `Unknown` | An optional, free-text profile field. The tweet is still valid and "Unknown" is explicit and easy to filter. |
| `user_timezone` | `Unknown` | Same reasoning as location. No value is invented. |

In [11]:
# Confirm the structural pattern before filling: only non-negative tweets lack a reason
is_negative_raw = clean_df["airline_sentiment"].str.strip().str.lower() == "negative"
print(f"Negative tweets missing negativereason: {clean_df.loc[is_negative_raw, 'negativereason'].isna().sum():,}")
print(f"Non-negative tweets missing negativereason: {clean_df.loc[~is_negative_raw, 'negativereason'].isna().sum():,}")

Negative tweets missing negativereason: 0
Non-negative tweets missing negativereason: 5,401


In [12]:
missing_before_fill = clean_df.isna().sum()

clean_df["negativereason"] = clean_df["negativereason"].fillna("Not Applicable")
clean_df["negativereason_confidence"] = clean_df["negativereason_confidence"].fillna(0)
clean_df["tweet_location"] = clean_df["tweet_location"].fillna("Unknown")
clean_df["user_timezone"] = clean_df["user_timezone"].fillna("Unknown")

filled_summary = missing_before_fill[missing_before_fill > 0].to_frame("values_filled")
filled_summary

,values_filled
negativereason,5401
negativereason_confidence,4075
tweet_location,4687
user_timezone,4775


## 7. Verify Missing Values

In [13]:
remaining_missing = clean_df.isna().sum()
display(remaining_missing.to_frame("missing_values"))
print(f"Total remaining missing values: {remaining_missing.sum():,}")

,missing_values
tweet_id,0
airline_sentiment,0
airline_sentiment_confidence,0
negativereason,0
negativereason_confidence,0
airline,0
retweet_count,0
text,0
tweet_created,0
tweet_location,0


Total remaining missing values: 0


## 8. Data Type Correction

| Column | From | To | Why |
|---|---|---|---|
| `tweet_id` | int64 | string | It is an identifier, not a quantity. Text avoids precision loss in spreadsheet and BI tools. |
| `airline_sentiment_confidence` | float | float | Enforced explicitly. |
| `negativereason_confidence` | float | float | Enforced explicitly after the fill. |
| `retweet_count` | int | int | Enforced explicitly. |
| `tweet_created` | text | datetime (UTC) | Enables date and time features. |

In [14]:
clean_df["tweet_id"] = clean_df["tweet_id"].astype("string")
clean_df["airline_sentiment_confidence"] = clean_df["airline_sentiment_confidence"].astype(float)
clean_df["negativereason_confidence"] = clean_df["negativereason_confidence"].astype(float)
clean_df["retweet_count"] = clean_df["retweet_count"].astype(int)

clean_df["tweet_created"] = pd.to_datetime(
    clean_df["tweet_created"],
    errors="coerce",
    utc=True
)

print(f"Invalid (unparseable) timestamps: {clean_df['tweet_created'].isna().sum():,}")
clean_df.dtypes.to_frame("dtype")

Invalid (unparseable) timestamps: 0


,dtype
tweet_id,string
airline_sentiment,str
airline_sentiment_confidence,float64
negativereason,str
negativereason_confidence,float64
airline,str
retweet_count,int64
text,str
tweet_created,"datetime64[us, UTC]"
tweet_location,str


## 9. Standardize Categories

- `airline_sentiment`: `positive` / `neutral` / `negative` → `Positive` / `Neutral` / `Negative`
- `airline`: strip leading and trailing whitespace
- `negativereason`: strip whitespace and rename `longlines` → `Long Lines` to match the Title Case style of the other categories. No categories are merged, so the meaning is unchanged.

In [15]:
sentiment_label_map = {"positive": "Positive", "neutral": "Neutral", "negative": "Negative"}
clean_df["airline_sentiment"] = (
    clean_df["airline_sentiment"].str.strip().str.lower().map(sentiment_label_map)
)

clean_df["airline"] = clean_df["airline"].str.strip()

negative_reason_map = {"longlines": "Long Lines"}
clean_df["negativereason"] = clean_df["negativereason"].str.strip().replace(negative_reason_map)

# Any label not covered by the map would become NaN; confirm none did
print(f"Unmapped sentiment labels: {clean_df['airline_sentiment'].isna().sum()}")
print(f"Sentiment categories: {sorted(clean_df['airline_sentiment'].unique())}")
print(f"Airlines: {sorted(clean_df['airline'].unique())}")
print(f"Negative reasons: {sorted(clean_df['negativereason'].unique())}")

Unmapped sentiment labels: 0
Sentiment categories: ['Negative', 'Neutral', 'Positive']
Airlines: ['American', 'Delta', 'Southwest', 'US Airways', 'United', 'Virgin America']
Negative reasons: ['Bad Flight', "Can't Tell", 'Cancelled Flight', 'Customer Service Issue', 'Damaged Luggage', 'Flight Attendant Complaints', 'Flight Booking Problems', 'Late Flight', 'Long Lines', 'Lost Luggage', 'Not Applicable']


### 9.1 Correct the Airline Label

Notebook 01 showed that the rows labelled `Delta` are a JetBlue collection: 2,218 of 2,222 are addressed to @JetBlue, while every other airline's rows mention that airline's own handle 100% of the time.

**Rule** (`src.data.correct_airline_labels`): a row labelled `Delta` whose text mentions `@jetblue` is relabelled `JetBlue`. The original value is kept in `airline_source`, so the correction is transparent and reversible.

Rows labelled `Delta` that do **not** mention @JetBlue cannot be verified: two address @DeltaAssist and two are outbound replies with no airline handle. They are excluded, because a two-tweet "Delta" group would be meaningless in airline comparisons and would break the expected-count assumption of the chi-square test in notebook 05.

In [16]:
clean_df = correct_airline_labels(clean_df)
unverified = clean_df[~clean_df["airline_verified"]]

print("Rows relabelled Delta -> JetBlue:", int(((clean_df["airline_source"] == "Delta") & (clean_df["airline"] == "JetBlue")).sum()))
print("Rows with an unverifiable airline (excluded):", len(unverified))
display(unverified[["tweet_id", "airline_source", "airline_sentiment", "text"]].assign(text=unverified["text"].map(mask_user_handles)))

rows_before_airline_check = len(clean_df)
clean_df = clean_df[clean_df["airline_verified"]].drop(columns="airline_verified").reset_index(drop=True)
print(f"Rows after excluding unverified airline labels: {len(clean_df):,}")

Rows relabelled Delta -> JetBlue: 2218
Rows with an unverifiable airline (excluded): 4


,tweet_id,airline_source,airline_sentiment,text
7330,569667821455708160,Delta,Neutral,"@user If you want, we can keep you updated on who's wearing who, &amp; who's with who, &amp; who wins what. We're ni..."
8215,568602015456215040,Delta,Positive,@user We think it's a treat to have you onboard! Enjoy your flight. 💙
11965,570294451261874177,Delta,Negative,@DeltaAssist now at 57 minutes waiting on Silver Elite line for someone to pick up! Help!
11966,568757671819661314,Delta,Negative,@DeltaAssist what I have to say is more than 140 characters! Plus you don't follow me


Rows after excluding unverified airline labels: 14,481


## 10. Feature Engineering

### 10.1 Date and time features

`tweet_created` is stored in UTC. Every raw timestamp carries the **same** `-0800` offset (checked below). Calendar features are therefore derived in that original UTC−08:00 clock time. This matters: deriving them from UTC would move every tweet posted after 16:00 into the next day and distort the date, weekday and hour analysis.

`Etc/GMT+8` is the standard IANA name for a fixed UTC−08:00 offset (the sign is inverted by convention).

In [17]:
# Verify the assumption: one single UTC offset in the raw timestamps
raw_offsets = df["tweet_created"].str[-5:].unique()
print(f"UTC offsets present in the raw file: {list(raw_offsets)}")
assert list(raw_offsets) == ["-0800"], "Unexpected offsets - revisit the time-zone logic"

SOURCE_TIMEZONE = "Etc/GMT+8"  # fixed UTC-08:00
created_source_time = clean_df["tweet_created"].dt.tz_convert(SOURCE_TIMEZONE)

UTC offsets present in the raw file: ['-0800']


In [18]:
clean_df["tweet_date"] = created_source_time.dt.date
clean_df["year"] = created_source_time.dt.year
clean_df["month"] = created_source_time.dt.month_name()
clean_df["month_number"] = created_source_time.dt.month
clean_df["day"] = created_source_time.dt.day
clean_df["day_name"] = created_source_time.dt.day_name()
clean_df["hour"] = created_source_time.dt.hour

clean_df[["tweet_created", "tweet_date", "year", "month", "month_number", "day", "day_name", "hour"]].head()

,tweet_created,tweet_date,year,month,month_number,day,day_name,hour
0,2015-02-24 19:35:52+00:00,2015-02-24,2015,February,2,24,Tuesday,11
1,2015-02-24 19:15:59+00:00,2015-02-24,2015,February,2,24,Tuesday,11
2,2015-02-24 19:15:48+00:00,2015-02-24,2015,February,2,24,Tuesday,11
3,2015-02-24 19:15:36+00:00,2015-02-24,2015,February,2,24,Tuesday,11
4,2015-02-24 19:14:45+00:00,2015-02-24,2015,February,2,24,Tuesday,11


### 10.2 Sentiment features

- `sentiment_score`: Positive = 1, Neutral = 0, Negative = −1. Its average gives a single comparable number per airline.
- `is_positive`, `is_negative`, `is_neutral`: 1/0 flags, so a mean of a flag is directly a percentage.

No engagement metrics (likes, impressions, comments) are created, because the dataset does not contain them.

In [19]:
sentiment_score_map = {"Positive": 1, "Neutral": 0, "Negative": -1}
clean_df["sentiment_score"] = clean_df["airline_sentiment"].map(sentiment_score_map).astype(int)

clean_df["is_positive"] = (clean_df["airline_sentiment"] == "Positive").astype(int)
clean_df["is_negative"] = (clean_df["airline_sentiment"] == "Negative").astype(int)
clean_df["is_neutral"] = (clean_df["airline_sentiment"] == "Neutral").astype(int)

# Each tweet must have exactly one flag set
flag_total = clean_df[["is_positive", "is_negative", "is_neutral"]].sum(axis=1)
print(f"Rows where flags do not sum to 1: {(flag_total != 1).sum()}")

clean_df[["airline_sentiment", "sentiment_score", "is_positive", "is_negative", "is_neutral"]].drop_duplicates()

Rows where flags do not sum to 1: 0


,airline_sentiment,sentiment_score,is_positive,is_negative,is_neutral
0,Neutral,0,0,0,1
1,Positive,1,1,0,0
3,Negative,-1,0,1,0


## 11. Final Data Validation

Every check below is computed on the final table. `assert` statements stop the notebook if any rule is broken.

In [20]:
expected_sentiments = {"Positive", "Neutral", "Negative"}

validation_results = {
    "Final rows": f"{clean_df.shape[0]:,}",
    "Final columns": clean_df.shape[1],
    "Null values": int(clean_df.isna().sum().sum()),
    "Exact duplicates": int(clean_df.duplicated().sum()),
    "Duplicate tweet IDs": int(clean_df.duplicated(subset="tweet_id").sum()),
    "Invalid dates": int(clean_df["tweet_created"].isna().sum()),
    "Negative retweet values": int((clean_df["retweet_count"] < 0).sum()),
    "Sentiment categories": sorted(clean_df["airline_sentiment"].unique()),
    "Airlines": clean_df["airline"].nunique(),
    "Negative reasons (incl. Not Applicable)": clean_df["negativereason"].nunique(),
    "airline_sentiment_confidence range": f"{clean_df['airline_sentiment_confidence'].min():.4f} - {clean_df['airline_sentiment_confidence'].max():.4f}",
    "negativereason_confidence range": f"{clean_df['negativereason_confidence'].min():.4f} - {clean_df['negativereason_confidence'].max():.4f}",
    "Date range (UTC-08:00)": f"{clean_df['tweet_date'].min()} to {clean_df['tweet_date'].max()}",
}

for check, result in validation_results.items():
    print(f"{check:<42}: {result}")

Final rows                                : 14,481
Final columns                             : 23
Null values                               : 0
Exact duplicates                          : 0
Duplicate tweet IDs                       : 0
Invalid dates                             : 0
Negative retweet values                   : 0
Sentiment categories                      : ['Negative', 'Neutral', 'Positive']
Airlines                                  : 6
Negative reasons (incl. Not Applicable)   : 11
airline_sentiment_confidence range        : 0.3350 - 1.0000
negativereason_confidence range           : 0.0000 - 1.0000
Date range (UTC-08:00)                    : 2015-02-16 to 2015-02-24


In [21]:
# Hard validation rules
assert clean_df.isna().sum().sum() == 0
assert clean_df.duplicated().sum() == 0
assert clean_df["tweet_id"].is_unique
assert set(clean_df["airline_sentiment"]) == expected_sentiments
assert (clean_df["retweet_count"] >= 0).all()
assert clean_df["airline_sentiment_confidence"].between(0, 1).all()
assert clean_df["negativereason_confidence"].between(0, 1).all()
# Business rule: only negative tweets can carry a complaint reason
assert (clean_df.loc[clean_df["airline_sentiment"] != "Negative", "negativereason"] == "Not Applicable").all()
assert (clean_df.loc[clean_df["airline_sentiment"] == "Negative", "negativereason"] != "Not Applicable").all()

print("All validation checks passed.")

All validation checks passed.


In [22]:
display(clean_df.dtypes.to_frame("dtype"))
display(clean_df["airline_sentiment"].value_counts().to_frame("tweets"))
display(clean_df["airline"].value_counts().to_frame("tweets"))
display(clean_df["negativereason"].value_counts().to_frame("tweets"))

,dtype
tweet_id,string
airline_sentiment,str
airline_sentiment_confidence,float64
negativereason,str
negativereason_confidence,float64
airline,str
retweet_count,int64
text,str
tweet_created,"datetime64[us, UTC]"
tweet_location,str


,tweets
airline_sentiment,
Negative,9082
Neutral,3067
Positive,2332


,tweets
airline,
United,3822
US Airways,2913
American,2604
Southwest,2420
JetBlue,2218
Virgin America,504


,tweets
negativereason,
Not Applicable,5399
Customer Service Issue,2883
Late Flight,1652
Can't Tell,1182
Cancelled Flight,831
Lost Luggage,718
Bad Flight,573
Flight Booking Problems,520
Flight Attendant Complaints,474


### 11.1 Airline Label Consistency (after correction)

Each airline's tweets should now mention that airline's own handle. This re-runs the notebook 01 check on the cleaned data and turns it into an assertion.

In [23]:
own_handles = {
    "Virgin America": "@virginamerica", "United": "@united", "Southwest": "@southwestair",
    "JetBlue": "@jetblue", "US Airways": "@usairways", "American": "@americanair",
}
lower_text = clean_df["text"].str.lower()
handle_share = pd.Series({
    airline: lower_text[clean_df["airline"] == airline].str.contains(handle, regex=False).mean() * 100
    for airline, handle in own_handles.items()
}).round(2)
display(handle_share.to_frame("mentions_own_handle_pct"))
assert (handle_share == 100).all(), "Some tweets do not mention their airline's handle"
assert set(clean_df["airline"]) == set(own_handles)

,mentions_own_handle_pct
Virgin America,100.0
United,100.0
Southwest,100.0
JetBlue,100.0
US Airways,100.0
American,100.0


### Cleaning Summary

In [24]:
raw_rows, clean_rows = len(df), len(clean_df)

print(f"Raw rows:                  {raw_rows:,}")
print(f"Exact duplicates removed:  {rows_before - rows_after:,}")
print(f"Duplicate IDs resolved:    {rows_before_id_resolution - rows_before_airline_check:,}")
print(f"Unverified airline labels: {rows_before_airline_check - clean_rows:,}")
print(f"Clean rows:                {clean_rows:,} ({clean_rows / raw_rows * 100:.2f}% of raw rows retained)")
print(f"Columns: {df.shape[1]} raw -> {clean_df.shape[1]} clean "
      f"({len(columns_to_remove)} removed, {clean_df.shape[1] - (df.shape[1] - len(columns_to_remove))} added)")

Raw rows:                  14,640
Exact duplicates removed:  39
Duplicate IDs resolved:    116
Unverified airline labels: 4
Clean rows:                14,481 (98.91% of raw rows retained)
Columns: 15 raw -> 23 clean (4 removed, 12 added)


**What was done**

1. Removed 4 columns that were near-empty or personal identifiers, each with a documented reason.
2. Identified tweet IDs with conflicting sentiment labels before de-duplicating.
3. Removed exact duplicate rows, then kept the single highest-confidence record for each repeated `tweet_id`.
4. Filled structurally missing values with meaningful labels instead of dropping rows.
5. Corrected data types (`tweet_id` as text, `tweet_created` as a UTC datetime).
6. Standardised category labels without merging or changing any category.
7. Engineered calendar features (in the source's UTC−08:00 clock time), a numeric sentiment score and 1/0 sentiment flags.
8. Validated the result with explicit assertions.
9. Corrected a demonstrable source labelling error: rows labelled `Delta` that are addressed to @JetBlue are relabelled `JetBlue` (original kept in `airline_source`); 4 rows whose airline could not be verified were excluded.

No synthetic rows, engagement metrics or model-generated labels were added.

## 12. Export

The cleaned table is saved to `data/processed/cleaned_tweets.csv`, the single input for notebooks 03–10 and for `python -m src.train`.

In [25]:
CLEAN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(CLEAN_DATA_PATH, index=False)

print(f"Cleaned dataset saved to: {CLEAN_DATA_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Exported: {len(clean_df):,} rows x {clean_df.shape[1]} columns")

Cleaned dataset saved to: data/processed/cleaned_tweets.csv
Exported: 14,481 rows x 23 columns


In [26]:
# Round-trip check: reload the export and confirm nothing was lost
reloaded = pd.read_csv(CLEAN_DATA_PATH, dtype={"tweet_id": "string"})
print(f"Reloaded shape matches: {reloaded.shape == clean_df.shape}")
print(f"Reloaded tweet IDs unique: {reloaded['tweet_id'].is_unique}")
print(f"Raw file untouched: {load_raw_tweets().shape == df.shape}")

Reloaded shape matches: True
Reloaded tweet IDs unique: True
Raw file untouched: True
